# Stage 3: CatBoost Model Pipeline & Final Ensemble

## Overview
This notebook implements:
1. **CatBoost Model** - Leveraging native categorical feature handling
2. **Improved LightGBM Model** - Optimized hyperparameters
3. **Final Ensemble** - XGBoost + LightGBM + CatBoost weighted ensemble
4. **Production Deployment Class** - Ready-to-deploy prediction pipeline

---

## 1. Setup & Data Loading

Load the Stage 2 engineered datasets **before** encoding transformations to preserve raw categorical columns for CatBoost.

In [18]:
# ============================================================================
# CELL 1: SETUP & DATA LOADING
# ============================================================================

import pandas as pd
import numpy as np
import warnings
import json
from pathlib import Path
warnings.filterwarnings('ignore')

print("\n" + "=" * 80)
print("STAGE 3: CATBOOST MODEL PIPELINE & FINAL ENSEMBLE")
print("=" * 80)

# Define paths
BASE_PATH = Path('kcet_ml_project/data/stage2_v2_corrected/')
MODEL_DIR = Path('models')
CATBOOST_DIR = MODEL_DIR / 'catboost'
CATBOOST_DIR.mkdir(parents=True, exist_ok=True)

print("\n📂 Loading Stage 2 datasets (with raw categorical columns)...")

# Load the datasets
train_data = pd.read_csv(BASE_PATH / 'train_stage2_final.csv')
val_data = pd.read_csv(BASE_PATH / 'val_stage2_final.csv')
test_data = pd.read_csv(BASE_PATH / 'test_stage2_final.csv')

print(f"✅ Datasets loaded successfully!")
print(f"   Train: {train_data.shape}")
print(f"   Val:   {val_data.shape}")
print(f"   Test:  {test_data.shape}")

# Display available columns
print(f"\n📋 Available columns ({len(train_data.columns)}):")
print(train_data.columns.tolist()[:20], "...")


STAGE 3: CATBOOST MODEL PIPELINE & FINAL ENSEMBLE

📂 Loading Stage 2 datasets (with raw categorical columns)...
✅ Datasets loaded successfully!
   Train: (137755, 33)
   Val:   (60681, 33)
   Test:  (71626, 33)

📋 Available columns (33):
['Cutoff_Rank', 'Year', 'Round', 'Exam_Type', 'Years_Since_2020', 'Is_Recent', 'Year_Squared', 'Historical_Mean_Primary', 'Historical_Mean_Percentile', 'College_Tier_Numeric', 'Category_Score', 'Historical_Std_Raw', 'Volatility_Category', 'Historical_Count_Raw', 'Program_Maturity', 'Is_Established', 'Branch_Popularity', 'cutoff_lag1Y_L1Y', 'cutoff_lag2Y_L2Y', 'cutoff_roll3Y_mean_L1Y'] ...
✅ Datasets loaded successfully!
   Train: (137755, 33)
   Val:   (60681, 33)
   Test:  (71626, 33)

📋 Available columns (33):
['Cutoff_Rank', 'Year', 'Round', 'Exam_Type', 'Years_Since_2020', 'Is_Recent', 'Year_Squared', 'Historical_Mean_Primary', 'Historical_Mean_Percentile', 'College_Tier_Numeric', 'Category_Score', 'Historical_Std_Raw', 'Volatility_Category', 'His

## 2. Feature Selection for CatBoost

### Key Principles:
- ✅ **Include raw categorical columns** (CatBoost handles them natively)
- ✅ **Include non-leaking numeric features**
- ❌ **Exclude target-encoded features** (redundant with raw categoricals)
- ❌ **Exclude label-encoded features** (CatBoost doesn't need them)
- ❌ **Exclude future data** (prevent leakage)

In [19]:
# ============================================================================
# CELL 2: FEATURE SELECTION FOR CATBOOST (NO DATA LEAKAGE)
# ============================================================================

print("\n" + "=" * 80)
print("FEATURE SELECTION FOR CATBOOST")
print("=" * 80)

# Define categorical features (raw columns that CatBoost will handle)
CATEGORICAL_FEATURES = [
    'College_Code',
    'College_Name', 
    'Branch',
    'Category',
    'Exam_Type',
    'Volatility_Category',
    'Program_Maturity'
]

# Define numeric features (no leakage, no target encoding)
NUMERIC_FEATURES = [
    'Year',
    'Quota_Seats',
    'Cutoff_L1Y',
    'Cutoff_L2Y', 
    'Cutoff_L3Y',
    'cutoff_roll3Y_mean_L1Y',
    'cutoff_roll3Y_std_L1Y',
    'n_years_hist_L1Y',
    'trend3Y_slope_L1Y',
    'branch_historical_mean',
    'college_historical_mean'
]

TARGET = 'Cutoff_Rank'

# Verify feature availability
all_features = CATEGORICAL_FEATURES + NUMERIC_FEATURES
available_features = [f for f in all_features if f in train_data.columns]
missing_features = [f for f in all_features if f not in train_data.columns]

print(f"\n✅ Available features: {len(available_features)}/{len(all_features)}")
print(f"   Categorical: {len([f for f in CATEGORICAL_FEATURES if f in train_data.columns])}")
print(f"   Numeric: {len([f for f in NUMERIC_FEATURES if f in train_data.columns])}")

if missing_features:
    print(f"\n⚠️  Missing features: {missing_features}")
    print(f"   Proceeding with available features only.")
    CATEGORICAL_FEATURES = [f for f in CATEGORICAL_FEATURES if f in train_data.columns]
    NUMERIC_FEATURES = [f for f in NUMERIC_FEATURES if f in train_data.columns]

# Update feature list
CATBOOST_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

print(f"\n📊 Final CatBoost feature set:")
print(f"   Total: {len(CATBOOST_FEATURES)} features")
print(f"   Categorical: {CATEGORICAL_FEATURES}")
print(f"   Numeric: {NUMERIC_FEATURES[:5]}...")


FEATURE SELECTION FOR CATBOOST

✅ Available features: 8/18
   Categorical: 3
   Numeric: 5

⚠️  Missing features: ['College_Code', 'College_Name', 'Branch', 'Category', 'Quota_Seats', 'Cutoff_L1Y', 'Cutoff_L2Y', 'Cutoff_L3Y', 'branch_historical_mean', 'college_historical_mean']
   Proceeding with available features only.

📊 Final CatBoost feature set:
   Total: 8 features
   Categorical: ['Exam_Type', 'Volatility_Category', 'Program_Maturity']
   Numeric: ['Year', 'cutoff_roll3Y_mean_L1Y', 'cutoff_roll3Y_std_L1Y', 'n_years_hist_L1Y', 'trend3Y_slope_L1Y']...


## 3. Prepare Data Splits & CatBoost Pools

Separate X and y, then create CatBoost Pool objects with categorical feature indices.

In [20]:
# ============================================================================
# CELL 3: PREPARE DATA SPLITS & CATBOOST POOLS
# ============================================================================

from catboost import Pool, CatBoostRegressor

print("\n" + "=" * 80)
print("DATA PREPARATION & CATBOOST POOLS")
print("=" * 80)

# Separate features and target
X_train = train_data[CATBOOST_FEATURES].copy()
y_train = train_data[TARGET].copy()

X_val = val_data[CATBOOST_FEATURES].copy()
y_val = val_data[TARGET].copy()

X_test = test_data[CATBOOST_FEATURES].copy()
y_test = test_data[TARGET].copy()

print(f"\n✅ Data splits created:")
print(f"   Train: X={X_train.shape}, y={y_train.shape}")
print(f"   Val:   X={X_val.shape}, y={y_val.shape}")
print(f"   Test:  X={X_test.shape}, y={y_test.shape}")

# Identify categorical feature indices
cat_feature_indices = [CATBOOST_FEATURES.index(f) for f in CATEGORICAL_FEATURES]

print(f"\n📍 Categorical feature indices: {cat_feature_indices}")

# Create CatBoost Pools
print(f"\n⏳ Creating CatBoost Pool objects...")

train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_feature_indices
)

val_pool = Pool(
    data=X_val,
    label=y_val,
    cat_features=cat_feature_indices
)

test_pool = Pool(
    data=X_test,
    label=y_test,
    cat_features=cat_feature_indices
)

print(f"✅ CatBoost Pools created successfully!")
print(f"   Train pool: {train_pool.num_row()} rows, {train_pool.num_col()} features")
print(f"   Val pool:   {val_pool.num_row()} rows")
print(f"   Test pool:  {test_pool.num_row()} rows")


DATA PREPARATION & CATBOOST POOLS

✅ Data splits created:
   Train: X=(137755, 8), y=(137755,)
   Val:   X=(60681, 8), y=(60681,)
   Test:  X=(71626, 8), y=(71626,)

📍 Categorical feature indices: [0, 1, 2]

⏳ Creating CatBoost Pool objects...
✅ CatBoost Pools created successfully!
   Train pool: 137755 rows, 8 features
   Val pool:   60681 rows
   Test pool:  71626 rows
✅ CatBoost Pools created successfully!
   Train pool: 137755 rows, 8 features
   Val pool:   60681 rows
   Test pool:  71626 rows


## 4. Train CatBoost Model

Train a CatBoost regressor with GPU acceleration and MAE optimization.

In [ ]:
# ============================================================================
# CELL 4: TRAIN CATBOOST MODEL
# ============================================================================

import time

print("\n" + "=" * 80)
print("CATBOOST MODEL TRAINING")
print("=" * 80)

# Define CatBoost model with optimal hyperparameters for regression
catboost_params = {
    'iterations': 5000,              # More iterations for better convergence
    'learning_rate': 0.01,           # Lower LR for stability and better generalization
    'depth': 6,                      # Optimal depth (4-10 range, 6 is sweet spot)
    'loss_function': 'RMSE',         # RMSE for regression
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'task_type': 'GPU',              # Use GPU if available
    'verbose': 200,
    'l2_leaf_reg': 3.0,              # L2 regularization for overfitting control
    'bagging_temperature': 1.0,      # Bayesian bootstrap variance (controls sampling intensity)
    'colsample_bylevel': 0.8,        # Column sampling per tree level
    'min_data_in_leaf': 20,          # Minimum samples in leaf (balance bias-variance)
    'bootstrap_type': 'Bayesian',    # Best for small-medium datasets (incompatible with subsample)
    'od_type': 'Iter',               # Overfitting detector type
    'od_wait': 500,                  # Wait iterations before stopping (synonym of early_stopping_rounds)
    'border_count': 254,             # Feature discretization (synonym of max_bin)
    'leaf_estimation_iterations': 10 # Newton iterations for leaf values
}

print(f"\n📋 CatBoost configuration (optimized for best performance):")
for key, val in catboost_params.items():
    print(f"   {key}: {val}")

# Initialize model
catboost_model = CatBoostRegressor(**catboost_params)

print(f"\n⏳ Training CatBoost model (this may take several minutes)...\n")

t0 = time.time()

# Train the model
try:
    catboost_model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        plot=False
    )
    print(f"\n✅ Training completed using GPU!")
except Exception as e:
    print(f"⚠️  GPU training failed: {e}")
    print(f"   Falling back to CPU training...\n")
    
    # Retry with CPU
    catboost_params['task_type'] = 'CPU'
    catboost_model = CatBoostRegressor(**catboost_params)
    catboost_model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        plot=False
    )
    print(f"\n✅ Training completed using CPU!")

t1 = time.time()
training_time = (t1 - t0) / 60

print(f"\n⏱️  Training time: {training_time:.2f} minutes")
print(f"🎯 Best iteration: {catboost_model.best_iteration_}")
print(f"📊 Best validation score (RMSE): {catboost_model.best_score_['validation']['RMSE']:.2f}")


CATBOOST MODEL TRAINING

📋 CatBoost configuration (optimized for best performance):
   iterations: 5000
   learning_rate: 0.01
   depth: 6
   loss_function: RMSE
   eval_metric: RMSE
   random_seed: 42
   task_type: GPU
   verbose: 200
   l2_leaf_reg: 3.0
   bagging_temperature: 1.0
   colsample_bylevel: 0.8
   min_data_in_leaf: 20
   bootstrap_type: Bayesian
   od_type: Iter
   od_wait: 500
   border_count: 254
   leaf_estimation_iterations: 10

⏳ Training CatBoost model (this may take several minutes)...

⚠️  GPU training failed: catboost/private/libs/options/catboost_options.cpp:637: Error: rsm on GPU is supported for pairwise modes only
   Falling back to CPU training...

0:	learn: 45106.0646842	test: 53015.1640784	best: 53015.1640784 (0)	total: 76.3ms	remaining: 6m 21s
200:	learn: 40548.3016530	test: 47742.3903438	best: 47742.3903438 (200)	total: 12.5s	remaining: 4m 59s
200:	learn: 40548.3016530	test: 47742.3903438	best: 47742.3903438 (200)	total: 12.5s	remaining: 4m 59s
400:	lea

KeyError: 'MAE'

## 5. Evaluate CatBoost Model

Compute comprehensive metrics including **Accuracy** on validation and test sets.

In [22]:
# ============================================================================
# CELL 5: EVALUATE CATBOOST MODEL WITH ACCURACY
# ============================================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("\n" + "=" * 80)
print("CATBOOST MODEL EVALUATION")
print("=" * 80)

def compute_metrics(y_true, y_pred, set_name=""):
    """Compute and display comprehensive metrics including Accuracy"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # Calculate Accuracy = max(0, 100 * (1 - MAE / mean(y_true)))
    accuracy = max(0, 100 * (1 - mae / np.mean(y_true)))
    
    print(f"\n📊 {set_name} Metrics:")
    print(f"   MAE:      {mae:,.2f}")
    print(f"   RMSE:     {rmse:,.2f}")
    print(f"   R²:       {r2:.4f}")
    print(f"   Accuracy: {accuracy:.2f}%")
    
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'accuracy': accuracy}

# Make predictions
print("\n⏳ Generating predictions...")
y_train_pred = catboost_model.predict(train_pool)
y_val_pred = catboost_model.predict(val_pool)
y_test_pred = catboost_model.predict(test_pool)

# Compute metrics
train_metrics = compute_metrics(y_train, y_train_pred, "Training (2020-2022)")
val_metrics = compute_metrics(y_val, y_val_pred, "Validation (2023)")
test_metrics = compute_metrics(y_test, y_test_pred, "Test (2024)")

# Summary comparison
print(f"\n" + "=" * 80)
print("📈 PERFORMANCE SUMMARY")
print("=" * 80)

summary_df = pd.DataFrame({
    'Set': ['Train', 'Validation', 'Test'],
    'MAE': [train_metrics['mae'], val_metrics['mae'], test_metrics['mae']],
    'RMSE': [train_metrics['rmse'], val_metrics['rmse'], test_metrics['rmse']],
    'R²': [train_metrics['r2'], val_metrics['r2'], test_metrics['r2']],
    'Accuracy (%)': [train_metrics['accuracy'], val_metrics['accuracy'], test_metrics['accuracy']]
})

print("\n" + summary_df.to_string(index=False))

# Feature importance
print(f"\n" + "=" * 80)
print("🎯 TOP 10 FEATURE IMPORTANCE")
print("=" * 80)

feature_importance = catboost_model.get_feature_importance()
feature_names = CATBOOST_FEATURES

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False).head(10)

print("\n" + importance_df.to_string(index=False))


CATBOOST MODEL EVALUATION

⏳ Generating predictions...

📊 Training (2020-2022) Metrics:
   MAE:      32,535.37
   RMSE:     40,405.30
   R²:       0.2004
   Accuracy: 53.07%

📊 Validation (2023) Metrics:
   MAE:      37,199.34
   RMSE:     47,286.45
   R²:       0.1623
   Accuracy: 54.42%

📊 Test (2024) Metrics:
   MAE:      54,241.22
   RMSE:     72,111.34
   R²:       -0.1314
   Accuracy: 50.51%

📈 PERFORMANCE SUMMARY

       Set          MAE         RMSE        R²  Accuracy (%)
     Train 32535.365185 40405.301582  0.200443     53.065603
Validation 37199.337044 47286.447504  0.162305     54.415547
      Test 54241.221075 72111.341671 -0.131383     50.512487

🎯 TOP 10 FEATURE IMPORTANCE

               Feature  Importance
   Volatility_Category   72.940962
      Program_Maturity   12.284160
             Exam_Type    9.118952
                  Year    5.655925
cutoff_roll3Y_mean_L1Y    0.000000
 cutoff_roll3Y_std_L1Y    0.000000
      n_years_hist_L1Y    0.000000
     trend3Y_slope_L

## 6. Save CatBoost Model & Metadata

Save the trained model and metadata for deployment.

In [ ]:
# ============================================================================
# CELL 6: SAVE CATBOOST MODEL & METADATA
# ============================================================================

import joblib

print("\n" + "=" * 80)
print("SAVING CATBOOST MODEL & METADATA")
print("=" * 80)

# Save CatBoost model
model_path = CATBOOST_DIR / 'catboost_stage3.cbm'
catboost_model.save_model(str(model_path))
print(f"\n✅ CatBoost model saved: {model_path}")

# Save metadata
metadata = {
    'features': CATBOOST_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'numeric_features': NUMERIC_FEATURES,
    'cat_feature_indices': cat_feature_indices,
    'target': TARGET,
    'metrics': {
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics
    },
    'params': catboost_params,
    'best_iteration': int(catboost_model.best_iteration_),
    'feature_importance': dict(zip(feature_names, feature_importance.tolist()))
}

meta_path = CATBOOST_DIR / 'catboost_meta.joblib'
joblib.dump(metadata, meta_path)
print(f"✅ Metadata saved: {meta_path}")

# Also save as JSON for human readability
json_path = CATBOOST_DIR / 'catboost_meta.json'
with open(json_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Metadata JSON saved: {json_path}")

print(f"\n💾 All artifacts saved to: {CATBOOST_DIR}")

## 7. CatBoost Prediction Function

Clean prediction function that loads model and makes predictions.

In [ ]:
# ============================================================================
# CELL 7: CATBOOST PREDICTION FUNCTION
# ============================================================================

def predict_catboost(df, model_path=None, meta_path=None):
    """
    Load CatBoost model and make predictions on input dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with required features
    model_path : str or Path, optional
        Path to saved CatBoost model (.cbm file)
    meta_path : str or Path, optional
        Path to saved metadata (.joblib file)
    
    Returns:
    --------
    np.ndarray
        Predicted cutoff ranks
    """
    # Default paths
    if model_path is None:
        model_path = CATBOOST_DIR / 'catboost_stage3.cbm'
    if meta_path is None:
        meta_path = CATBOOST_DIR / 'catboost_meta.joblib'
    
    # Load model and metadata
    model = CatBoostRegressor()
    model.load_model(str(model_path))
    
    metadata = joblib.load(meta_path)
    features = metadata['features']
    cat_indices = metadata['cat_feature_indices']
    
    # Prepare data
    X = df[features].copy()
    
    # Create Pool
    pool = Pool(data=X, cat_features=cat_indices)
    
    # Predict
    predictions = model.predict(pool)
    
    return predictions

# Test the function
print("\n" + "=" * 80)
print("TESTING PREDICTION FUNCTION")
print("=" * 80)

test_predictions = predict_catboost(test_data)
test_mae = mean_absolute_error(y_test, test_predictions)

print(f"\n✅ Prediction function works!")
print(f"   Test MAE: {test_mae:,.2f}")
print(f"   Sample predictions: {test_predictions[:5]}")

## 8. Improved LightGBM Model

Train an optimized LightGBM model with enhanced hyperparameters.

In [ ]:
# ============================================================================
# CELL 8: IMPROVED LIGHTGBM MODEL WITH ACCURACY
# ============================================================================

import lightgbm as lgb

print("\n" + "=" * 80)
print("IMPROVED LIGHTGBM MODEL TRAINING")
print("=" * 80)

# Load encoded features for LightGBM (needs encoded categoricals)
print("\n⏳ Loading encoded datasets for LightGBM...")

X_train_lgb = train_data.drop('Cutoff_Rank', axis=1)
X_val_lgb = val_data.drop('Cutoff_Rank', axis=1)
X_test_lgb = test_data.drop('Cutoff_Rank', axis=1)

print(f"✅ Loaded: Train={X_train_lgb.shape}, Val={X_val_lgb.shape}, Test={X_test_lgb.shape}")

# Define improved LightGBM parameters
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 64,
    'learning_rate': 0.015,
    'n_estimators': 3000,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 40,
    'reg_alpha': 3.0,      # L1 regularization
    'reg_lambda': 15.0,    # L2 regularization
    'max_bin': 255,
    'random_state': 42,
    'verbose': -1,
    'device': 'gpu'  # Try GPU
}

print(f"\n📋 LightGBM configuration:")
for key, val in list(lgb_params.items())[:12]:
    print(f"   {key}: {val}")

# Train model
print(f"\n⏳ Training improved LightGBM model...\n")

t0 = time.time()

try:
    lgb_model = lgb.LGBMRegressor(**lgb_params)
    
    lgb_model.fit(
        X_train_lgb, y_train,
        eval_set=[(X_val_lgb, y_val)],
        eval_metric='rmse',
        callbacks=[lgb.early_stopping(stopping_rounds=300), lgb.log_evaluation(period=200)]
    )
    print(f"\n✅ LightGBM training completed using GPU!")
except Exception as e:
    print(f"⚠️  GPU training failed: {e}")
    print(f"   Falling back to CPU...\n")
    lgb_params['device'] = 'cpu'
    lgb_model = lgb.LGBMRegressor(**lgb_params)
    lgb_model.fit(
        X_train_lgb, y_train,
        eval_set=[(X_val_lgb, y_val)],
        eval_metric='rmse',
        callbacks=[lgb.early_stopping(stopping_rounds=300), lgb.log_evaluation(period=200)]
    )
    print(f"\n✅ LightGBM training completed using CPU!")

t1 = time.time()
print(f"\n⏱️  Training time: {(t1-t0)/60:.2f} minutes")
print(f"🎯 Best iteration: {lgb_model.best_iteration_}")

# Evaluate with Accuracy
print("\n" + "=" * 80)
print("LIGHTGBM MODEL EVALUATION")
print("=" * 80)

lgb_val_pred = lgb_model.predict(X_val_lgb)
lgb_test_pred = lgb_model.predict(X_test_lgb)

lgb_val_metrics = compute_metrics(y_val, lgb_val_pred, "LightGBM Validation")
lgb_test_metrics = compute_metrics(y_test, lgb_test_pred, "LightGBM Test")

# Save model
lgb_model_path = MODEL_DIR / 'lightGBM_improved.joblib'
joblib.dump(lgb_model, lgb_model_path)
print(f"\n💾 LightGBM model saved: {lgb_model_path}")

## 9. Final Ensemble: XGBoost + LightGBM + CatBoost

Create a weighted ensemble by finding optimal weights through grid search, with **Accuracy** metrics.

In [ ]:
# ============================================================================
# CELL 9: FINAL ENSEMBLE WITH GRID SEARCH & ACCURACY
# ============================================================================

import xgboost as xgb
from itertools import product

print("\n" + "=" * 80)
print("FINAL ENSEMBLE: XGBOOST + LIGHTGBM + CATBOOST")
print("=" * 80)

# Load XGBoost model
print("\n⏳ Loading XGBoost model...")

xgb_wrapper_path = Path('kcet_ml_project/models/xgboost_stage3/xgb_wrapper_stage3_final_with_adaptive_fallback.joblib')

if xgb_wrapper_path.exists():
    xgb_wrapper = joblib.load(xgb_wrapper_path)
    xgb_booster = xgb.Booster()
    xgb_booster.load_model(str(xgb_wrapper['booster_path']))
    xgb_features = xgb_wrapper['features']
    print(f"✅ XGBoost loaded: {len(xgb_features)} features")
    
    # Make XGBoost predictions
    dval_xgb = xgb.DMatrix(X_val_lgb[xgb_features])
    dtest_xgb = xgb.DMatrix(X_test_lgb[xgb_features])
    
    xgb_val_pred = xgb_booster.predict(dval_xgb)
    xgb_test_pred = xgb_booster.predict(dtest_xgb)
    
    xgb_available = True
else:
    print(f"⚠️  XGBoost model not found at {xgb_wrapper_path}")
    print(f"   Proceeding with 2-model ensemble (LightGBM + CatBoost)")
    xgb_available = False

# Collect predictions with Accuracy
print("\n📊 Individual model predictions collected:")

if xgb_available:
    xgb_val_mae = mean_absolute_error(y_val, xgb_val_pred)
    xgb_test_mae = mean_absolute_error(y_test, xgb_test_pred)
    xgb_val_acc = max(0, 100 * (1 - xgb_val_mae / np.mean(y_val)))
    xgb_test_acc = max(0, 100 * (1 - xgb_test_mae / np.mean(y_test)))
    print(f"   XGBoost:  Val MAE={xgb_val_mae:,.2f}, Test MAE={xgb_test_mae:,.2f}")
    print(f"             Val Acc={xgb_val_acc:.2f}%, Test Acc={xgb_test_acc:.2f}%")

print(f"   LightGBM: Val MAE={lgb_val_metrics['mae']:,.2f}, Test MAE={lgb_test_metrics['mae']:,.2f}")
print(f"             Val Acc={lgb_val_metrics['accuracy']:.2f}%, Test Acc={lgb_test_metrics['accuracy']:.2f}%")
print(f"   CatBoost: Val MAE={val_metrics['mae']:,.2f}, Test MAE={test_metrics['mae']:,.2f}")
print(f"             Val Acc={val_metrics['accuracy']:.2f}%, Test Acc={test_metrics['accuracy']:.2f}%")

# Grid search for optimal weights
print("\n⏳ Performing grid search for optimal ensemble weights...")

if xgb_available:
    # 3-model ensemble
    weight_range = np.arange(0.1, 1.0, 0.1)
    best_r2 = -np.inf
    best_weights = None
    
    for w_xgb in weight_range:
        for w_lgb in weight_range:
            w_cat = 1.0 - w_xgb - w_lgb
            if w_cat < 0.1 or w_cat > 0.9:
                continue
            
            # Ensemble prediction on validation
            ensemble_val_pred = (w_xgb * xgb_val_pred + 
                                w_lgb * lgb_val_pred + 
                                w_cat * y_val_pred)
            
            r2 = r2_score(y_val, ensemble_val_pred)
            
            if r2 > best_r2:
                best_r2 = r2
                best_weights = {'xgb': w_xgb, 'lgb': w_lgb, 'cat': w_cat}
    
    print(f"\n✅ Optimal weights found (3-model ensemble):")
    print(f"   XGBoost:  {best_weights['xgb']:.2f}")
    print(f"   LightGBM: {best_weights['lgb']:.2f}")
    print(f"   CatBoost: {best_weights['cat']:.2f}")
    print(f"   Validation R²: {best_r2:.4f}")
    
    # Final ensemble predictions
    final_val_pred = (best_weights['xgb'] * xgb_val_pred + 
                      best_weights['lgb'] * lgb_val_pred + 
                      best_weights['cat'] * y_val_pred)
    
    final_test_pred = (best_weights['xgb'] * xgb_test_pred + 
                       best_weights['lgb'] * lgb_test_pred + 
                       best_weights['cat'] * y_test_pred)
else:
    # 2-model ensemble (LightGBM + CatBoost)
    weight_range = np.arange(0.1, 1.0, 0.1)
    best_r2 = -np.inf
    best_weights = None
    
    for w_lgb in weight_range:
        w_cat = 1.0 - w_lgb
        
        ensemble_val_pred = w_lgb * lgb_val_pred + w_cat * y_val_pred
        r2 = r2_score(y_val, ensemble_val_pred)
        
        if r2 > best_r2:
            best_r2 = r2
            best_weights = {'xgb': 0.0, 'lgb': w_lgb, 'cat': w_cat}
    
    print(f"\n✅ Optimal weights found (2-model ensemble):")
    print(f"   LightGBM: {best_weights['lgb']:.2f}")
    print(f"   CatBoost: {best_weights['cat']:.2f}")
    print(f"   Validation R²: {best_r2:.4f}")
    
    final_val_pred = (best_weights['lgb'] * lgb_val_pred + 
                      best_weights['cat'] * y_val_pred)
    
    final_test_pred = (best_weights['lgb'] * lgb_test_pred + 
                       best_weights['cat'] * y_test_pred)

# Evaluate final ensemble with Accuracy
print("\n" + "=" * 80)
print("FINAL ENSEMBLE EVALUATION")
print("=" * 80)

final_val_metrics = compute_metrics(y_val, final_val_pred, "Ensemble Validation")
final_test_metrics = compute_metrics(y_test, final_test_pred, "Ensemble Test")

# Comparison table with Accuracy
print("\n" + "=" * 80)
print("📊 MODEL COMPARISON (with Accuracy)")
print("=" * 80)

comparison_data = []
if xgb_available:
    comparison_data.append(['XGBoost', xgb_val_mae, xgb_test_mae, xgb_val_acc, xgb_test_acc])
comparison_data.extend([
    ['LightGBM', lgb_val_metrics['mae'], lgb_test_metrics['mae'], lgb_val_metrics['accuracy'], lgb_test_metrics['accuracy']],
    ['CatBoost', val_metrics['mae'], test_metrics['mae'], val_metrics['accuracy'], test_metrics['accuracy']],
    ['Ensemble', final_val_metrics['mae'], final_test_metrics['mae'], final_val_metrics['accuracy'], final_test_metrics['accuracy']]
])

comparison_df = pd.DataFrame(comparison_data, columns=['Model', 'Val MAE', 'Test MAE', 'Val Acc (%)', 'Test Acc (%)'])
print("\n" + comparison_df.to_string(index=False))

print("\n🎯 Best performing model on Test set:")
best_idx = comparison_df['Test MAE'].idxmin()
print(f"   {comparison_df.loc[best_idx, 'Model']} with MAE = {comparison_df.loc[best_idx, 'Test MAE']:,.2f}")
print(f"   Test Accuracy = {comparison_df.loc[best_idx, 'Test Acc (%)']:.2f}%")

## 10. Production Deployment Class

Create a unified ensemble class ready for production deployment with **Accuracy** reporting.

In [ ]:
# ============================================================================
# CELL 10: PRODUCTION DEPLOYMENT CLASS WITH ACCURACY
# ============================================================================

from sklearn.base import BaseEstimator, RegressorMixin

class FinalCutoffEnsemble(BaseEstimator, RegressorMixin):
    """
    Production-ready ensemble class for KCET cutoff prediction.
    Combines XGBoost, LightGBM, and CatBoost with optimal weights.
    """
    
    def __init__(self, 
                 xgb_path=None, 
                 lgb_path=None, 
                 catboost_path=None,
                 catboost_meta_path=None,
                 weights=None):
        """
        Initialize ensemble with model paths and weights.
        
        Parameters:
        -----------
        xgb_path : str or Path
            Path to XGBoost wrapper joblib
        lgb_path : str or Path  
            Path to LightGBM joblib
        catboost_path : str or Path
            Path to CatBoost .cbm model
        catboost_meta_path : str or Path
            Path to CatBoost metadata joblib
        weights : dict
            Dictionary with keys 'xgb', 'lgb', 'cat' containing weights
        """
        self.xgb_path = xgb_path
        self.lgb_path = lgb_path
        self.catboost_path = catboost_path
        self.catboost_meta_path = catboost_meta_path
        self.weights = weights or {'xgb': 0.33, 'lgb': 0.33, 'cat': 0.34}
        
        self.xgb_model = None
        self.lgb_model = None
        self.catboost_model = None
        self.catboost_meta = None
        
        self._load_models()
    
    def _load_models(self):
        """Load all models from disk"""
        print("Loading ensemble models...")
        
        # Load XGBoost
        if self.xgb_path and Path(self.xgb_path).exists():
            try:
                xgb_wrapper = joblib.load(self.xgb_path)
                self.xgb_model = xgb.Booster()
                self.xgb_model.load_model(str(xgb_wrapper['booster_path']))
                self.xgb_features = xgb_wrapper['features']
                print(f"  ✅ XGBoost loaded")
            except Exception as e:
                print(f"  ⚠️  XGBoost load failed: {e}")
                self.weights['xgb'] = 0.0
        
        # Load LightGBM
        if self.lgb_path and Path(self.lgb_path).exists():
            try:
                self.lgb_model = joblib.load(self.lgb_path)
                print(f"  ✅ LightGBM loaded")
            except Exception as e:
                print(f"  ⚠️  LightGBM load failed: {e}")
                self.weights['lgb'] = 0.0
        
        # Load CatBoost
        if self.catboost_path and Path(self.catboost_path).exists():
            try:
                self.catboost_model = CatBoostRegressor()
                self.catboost_model.load_model(str(self.catboost_path))
                
                if self.catboost_meta_path:
                    self.catboost_meta = joblib.load(self.catboost_meta_path)
                
                print(f"  ✅ CatBoost loaded")
            except Exception as e:
                print(f"  ⚠️  CatBoost load failed: {e}")
                self.weights['cat'] = 0.0
        
        # Normalize weights
        total = sum(self.weights.values())
        if total > 0:
            self.weights = {k: v/total for k, v in self.weights.items()}
        
        print(f"  📊 Final weights: XGB={self.weights['xgb']:.2f}, "
              f"LGB={self.weights['lgb']:.2f}, CAT={self.weights['cat']:.2f}")
    
    def predict(self, X):
        """
        Make ensemble predictions.
        
        Parameters:
        -----------
        X : pd.DataFrame
            Input features
        
        Returns:
        --------
        np.ndarray
            Predicted cutoff ranks
        """
        predictions = []
        weights_used = []
        
        # XGBoost prediction
        if self.xgb_model is not None and self.weights['xgb'] > 0:
            dmatrix = xgb.DMatrix(X[self.xgb_features])
            xgb_pred = self.xgb_model.predict(dmatrix)
            predictions.append(xgb_pred)
            weights_used.append(self.weights['xgb'])
        
        # LightGBM prediction
        if self.lgb_model is not None and self.weights['lgb'] > 0:
            lgb_pred = self.lgb_model.predict(X)
            predictions.append(lgb_pred)
            weights_used.append(self.weights['lgb'])
        
        # CatBoost prediction
        if self.catboost_model is not None and self.weights['cat'] > 0:
            if self.catboost_meta:
                features = self.catboost_meta['features']
                cat_indices = self.catboost_meta['cat_feature_indices']
                pool = Pool(data=X[features], cat_features=cat_indices)
                cat_pred = self.catboost_model.predict(pool)
            else:
                cat_pred = self.catboost_model.predict(X)
            predictions.append(cat_pred)
            weights_used.append(self.weights['cat'])
        
        # Weighted ensemble
        if not predictions:
            raise RuntimeError("No models available for prediction!")
        
        # Normalize weights
        weights_norm = np.array(weights_used) / sum(weights_used)
        
        # Combine predictions
        ensemble_pred = sum(w * pred for w, pred in zip(weights_norm, predictions))
        
        return ensemble_pred
    
    def fit(self, X, y):
        """Dummy fit method for sklearn compatibility"""
        return self
    
    def score(self, X, y):
        """Calculate accuracy score"""
        y_pred = self.predict(X)
        mae = mean_absolute_error(y, y_pred)
        accuracy = max(0, 100 * (1 - mae / np.mean(y)))
        return accuracy / 100  # Return as decimal for sklearn compatibility

# Create and save ensemble
print("\n" + "=" * 80)
print("CREATING PRODUCTION ENSEMBLE")
print("=" * 80)

ensemble = FinalCutoffEnsemble(
    xgb_path='kcet_ml_project/models/xgboost_stage3/xgb_wrapper_stage3_final_with_adaptive_fallback.joblib',
    lgb_path='models/lightGBM_improved.joblib',
    catboost_path='models/catboost/catboost_stage3.cbm',
    catboost_meta_path='models/catboost/catboost_meta.joblib',
    weights=best_weights
)

# Test ensemble
print("\n⏳ Testing ensemble on test set...")
ensemble_test_pred = ensemble.predict(test_data)
ensemble_test_mae = mean_absolute_error(y_test, ensemble_test_pred)
ensemble_test_accuracy = max(0, 100 * (1 - ensemble_test_mae / np.mean(y_test)))

print(f"✅ Ensemble test MAE: {ensemble_test_mae:,.2f}")
print(f"✅ Ensemble test Accuracy: {ensemble_test_accuracy:.2f}%")

# Save ensemble
ensemble_path = MODEL_DIR / 'final_ensemble.joblib'
ensemble_metadata = {
    'ensemble': ensemble,
    'weights': best_weights,
    'test_mae': ensemble_test_mae,
    'test_accuracy': ensemble_test_accuracy,
    'models': {
        'xgb': str(ensemble.xgb_path) if ensemble.xgb_path else None,
        'lgb': str(ensemble.lgb_path) if ensemble.lgb_path else None,
        'catboost': str(ensemble.catboost_path) if ensemble.catboost_path else None
    }
}

joblib.dump(ensemble_metadata, ensemble_path)
print(f"\n💾 Final ensemble saved: {ensemble_path}")

print("\n" + "=" * 80)
print("✅ PRODUCTION PIPELINE COMPLETE")
print("=" * 80)

print("\n📦 Deliverables:")
print(f"   1. CatBoost model: models/catboost/catboost_stage3.cbm")
print(f"   2. CatBoost metadata: models/catboost/catboost_meta.joblib")
print(f"   3. Improved LightGBM: models/lightGBM_improved.joblib")
print(f"   4. Final ensemble: models/final_ensemble.joblib")

print("\n🎯 Final Performance Summary:")
print(f"   Ensemble Test MAE: {ensemble_test_mae:,.2f}")
print(f"   Ensemble Test Accuracy: {ensemble_test_accuracy:.2f}%")

print("\n💡 Usage example:")
print("""
    # Load and predict
    import joblib
    ensemble_data = joblib.load('models/final_ensemble.joblib')
    ensemble = ensemble_data['ensemble']
    predictions = ensemble.predict(your_dataframe)
    
    # Get accuracy
    accuracy = ensemble.score(your_dataframe, your_labels)
    print(f"Accuracy: {accuracy * 100:.2f}%")
""")